# 두 Med baseline 재현 · Colab Tesla T4

**Phi3V-Med / Phi3.5V-Med, arXiv:2507.09209v1.** RAG/Expert-CFG 제외.

현재 GPU 검증 전인 실행용 notebook입니다. 실제 benchmark 결과는 실행해서 생성해야 합니다. synthetic 테스트는 논문 재현 수치가 아닙니다.

1. Runtime → Change runtime type → T4 GPU.
2. 첫 셀이 GitHub에서 코드를 자동으로 내려받습니다. Drive는 데이터와 결과 보관용입니다.
3. 설치 셀이 별도 Python 3.11 환경을 만듭니다. Colab 기본 Python 버전은 바꾸지 않으며 런타임 재시작도 필요 없습니다.
4. Hugging Face 자동 데이터 준비 → smoke → 전체 test → 필요한 경우 downstream 학습.

자세한 프로토콜과 논문의 모든 실험 목록은 README와 docs/EXPERIMENT_AUDIT.md를 확인하세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # 이미 mounted라는 메시지는 정상입니다.
from pathlib import Path
import os
import subprocess
import sys
REPO = Path('/content/26fall_urp_expert_cfg')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'main',
                    'https://github.com/each-yi/26fall_urp_expert_cfg.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
assert (REPO / 'baseline/infer.py').is_file(), f'코드 다운로드를 확인하세요: {REPO}'
os.chdir(REPO)
# 재시작 후에도 notebook 커널에서 baseline 모듈을 찾도록 등록합니다.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('Repository:', REPO)


In [ ]:
# Colab 커널이 Python 3.13이어도 고정 stack을 Python 3.11 환경에 설치합니다.
# 이전 %pip 설치 셀은 사용하지 마세요. 설치 뒤 kernel restart는 필요 없습니다.
import subprocess
import sys
from pathlib import Path
REPO = Path('/content/26fall_urp_expert_cfg')
subprocess.run([sys.executable, str(REPO/'scripts/setup_colab.py')], check=True)
PYTHON = '/content/urp-venv/bin/python'
print('이후 모든 baseline 명령의 Python:', PYTHON)


## Python 3.11 전용 환경 준비 완료

이후 설치 패키지를 notebook kernel에 import하지 않고 `PYTHON` subprocess에서 실행합니다. Colab 기본 Python이 3.13이어도 PyArrow/NumPy/PyTorch의 소스 빌드가 발생하지 않습니다. 커널 재시작은 필요 없습니다.

FP16/NF4를 사용하며 T4에서 BF16/FlashAttention2를 켜지 않습니다. base/adapter는 원저자의 공개 Hugging Face 저장소에서 받습니다. 접근 오류를 임의의 다른 모델로 우회하지 않습니다.


In [ ]:
# GPU 검증도 설치된 Python 3.11에서 실행합니다.
check = """
import sys, torch, transformers, peft, bitsandbytes, datasets, pyarrow
assert sys.version_info[:2] == (3, 11), sys.version
assert torch.cuda.is_available(), 'T4 GPU runtime을 선택하세요.'
print('Python:', sys.version)
print('Versions:', torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)
print('Datasets / PyArrow:', datasets.__version__, pyarrow.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
assert torch.__version__.startswith('2.4.1') and transformers.__version__ == '4.44.2'
"""
subprocess.run([PYTHON, '-c', check], cwd=str(REPO), check=True)


In [ ]:
# CPU 평가 로직 검증. 합성 예측으로 보고서 경로를 검증하며 모델 재현 수치가 아닙니다.
import subprocess
result = subprocess.run(
    [PYTHON, '-u', '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(result.stdout)
if result.returncode:
    raise RuntimeError('CPU 테스트 실패: 위 출력의 마지막 traceback을 확인하세요.')


## Hugging Face 데이터 자동 준비

아래 셀이 세 dataset의 train/test(제공되는 경우 validation 포함)를 자동으로 다운로드하고, 이미지를 저장하고 JSONL로 변환합니다. 수동 다운로드/경로 수정은 필요 없습니다. SLAKE는 q_lang이 있는 경우 영어만 남깁니다.

* VQA-RAD: flaviagiammarino/vqa-rad
* SLAKE: BoKelvin/SLAKE
* PathVQA: flaviagiammarino/path-vqa

미러가 제공하는 split을 그대로 사용하며 임의 재분할하지 않습니다. 논문의 정확한 split과 동일하다는 보장은 없습니다. 소스 commit SHA, 실제 컬럼, 필터 전후 개수와 checksum을 provenance.json에 저장합니다. 준비가 끝난 데이터는 Drive에서 재사용합니다. HF 인증이 필요한 소스로 바뀌면 Colab의 HF_TOKEN secret 또는 huggingface_hub.login()을 사용하고 토큰을 코드에 쓰지 마세요.

이 개발 서버는 HF 접속이 차단되어 실제 다운로드가 미검증입니다. 소스 스키마가 다르거나 이미지가 제공되지 않으면 원인을 표시하고 멈추며, 다른 데이터셋으로 바꾸지 않습니다.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

DATA_ROOT = Path('/content/drive/MyDrive/urp_data/hf')
RUN_ROOT = Path('/content/drive/MyDrive/urp_runs')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
MODELS = ['phi3', 'phi35']
DATASETS = ['vqa_rad', 'slake', 'pathvqa']

# 최초 1회 자동 다운로드/변환; 이후 Drive의 완료 manifest와 파일을 검사하고 재사용합니다.
subprocess.run([
    PYTHON, '-m', 'baseline.download_data',
    '--output', str(DATA_ROOT), '--cache-dir', '/content/hf_datasets_cache',
    '--datasets', *DATASETS,
], cwd=str(REPO), check=True)
paths = json.loads((DATA_ROOT/'data_paths.json').read_text())
DATA = {ds: {key: Path(value) for key, value in paths[ds].items()} for ds in DATASETS}
from baseline.data import load_data
for ds in DATASETS:
    rows = load_data(DATA[ds]['test'], 'test')
    assert all((DATA[ds]['images']/r['image']).is_file() for r in rows), ds
    print(ds, len(rows), 'test QA', sum(r['answer_type']=='open' for r in rows), 'open')
    print('source / split provenance:', DATA_ROOT/ds/'provenance.json')


## 작은 GPU smoke 먼저

3개 질문의 실제 forward/generate를 확인합니다. 오류가 나면 전체 실험을 시작하지 마세요. crop16 OOM이면 런타임을 재시작하고 `CROPS=4`로 명시적으로 변경합니다. 낮춘 crop은 논문과 차이이며 결과 이름에도 반영합니다. 기본 최대128 생성 토큰은 긴 답에서 부족할 수 있습니다. report의 truncation 개수를 확인하세요.


In [ ]:
CROPS = 16
PRECISION = '4bit'
MAX_NEW_TOKENS = 128

def run_module(module, *args):
    subprocess.run([PYTHON, '-m', module, *map(str, args)], cwd=str(REPO), check=True)

ds = DATASETS[0]
smoke = RUN_ROOT / f'smoke_phi35_{ds}_{PRECISION}_crop{CROPS}.jsonl'
run_module('baseline.infer', '--model', 'phi35', '--setting', 'zero_med',
           '--data', DATA[ds]['test'], '--image-root', DATA[ds]['images'],
           '--crops', CROPS, '--precision', PRECISION, '--max-new-tokens', MAX_NEW_TOKENS,
           '--limit', 3, '--output', smoke)
run_module('baseline.evaluate', '--data', DATA[ds]['test'], '--predictions', smoke,
           '--output', RUN_ROOT/'smoke_report')
# coverage 3/N은 정상이며 전체 benchmark 결과로 해석하면 안 됩니다.


## 두 Med 모델 × 세 dataset zero-shot 전체 평가

공개 의료 adapter를 재사용합니다. **표3 Closed**에만 직접 대응하며 표1 downstream 수치와 혼동하지 않습니다. 선택적으로 `LABEL_SAMPLES=10`, `SELF_CHECK=True`를 켜서 부록의 대안 confidence도 수집할 수 있습니다. 비용이 크게 증가합니다. 논문과 동일한 calibration 규칙이 공개되지 않아 표6은 잠정 비교입니다.

같은 output 경로에서 동일 command를 다시 실행하면 완료된 질문을 건너뜁니다. 모델/데이터/옵션이 바뀌면 새 경로가 필요합니다. 처음 실행한 meta.json의 model_revision/adapter_revision SHA를 확인하고 이후 실험에 고정하세요.


In [ ]:
LABEL_SAMPLES = 0  # 부록 label probability 비교 시 10
SELF_CHECK = False  # zero-shot Is True? 비교 시 True
for model in MODELS:
    for ds in DATASETS:
        output = RUN_ROOT / f'{model}_zero_med_{ds}_{PRECISION}_crop{CROPS}.jsonl'
        extra = ['--self-check'] if SELF_CHECK else []
        run_module('baseline.infer', '--model', model, '--setting', 'zero_med',
                   '--data', DATA[ds]['test'], '--image-root', DATA[ds]['images'],
                   '--crops', CROPS, '--precision', PRECISION, '--max-new-tokens', MAX_NEW_TOKENS,
                   '--label-samples', LABEL_SAMPLES, '--output', output, *extra)
        run_module('baseline.evaluate', '--data', DATA[ds]['test'], '--predictions', output,
                   '--output', RUN_ROOT/f'{output.stem}_report', '--plots')


## 표1: dataset별 downstream 학습 → 평가

6개 별도 학습은 여러 T4 세션이 걸릴 수 있습니다. 무료 Colab 시간 안에 완료를 보장할 수 없습니다. 공개 medical LoRA부터 QLoRA로 계속 학습하며, 논문의 647k alignment/instruction 두 단계 전체를 다시 학습하지 않습니다. Drive checkpoint로 세션 중단을 견딥니다.

선택한 한 model/dataset부터 실행합니다. rank64, LR1e-4, effective batch64,10epoch. 기본 train crops4는 T4 절충이고 inference crop16과 다릅니다. 여유가 있으면 train도16을 시도하세요. 이 차이와 NF4/FP16은 기록됩니다. train 명령이 완료된 adapter만 평가할 수 있습니다.


In [ ]:
# 이 셀을 실행하려면 RUN_TRAIN=True로 설정하세요.
RUN_TRAIN = False
TRAIN_MODEL = 'phi35'  # 이후 phi3도 반복
TRAIN_DATASET = 'vqa_rad'  # 이후 slake, pathvqa도 반복
TRAIN_CROPS = 4
RESUME = False
adapter = RUN_ROOT / f'{TRAIN_MODEL}_{TRAIN_DATASET}_downstream_traincrop{TRAIN_CROPS}'
if RUN_TRAIN:
    extra = ['--resume'] if RESUME else []
    run_module('baseline.train', '--model', TRAIN_MODEL,
               '--train', DATA[TRAIN_DATASET]['train'], '--test', DATA[TRAIN_DATASET]['test'],
               '--image-root', DATA[TRAIN_DATASET]['images'], '--crops', TRAIN_CROPS,
               '--output', adapter, *extra)
else:
    print('Downstream 학습 미실행. 표1 재현을 주장할 수 없습니다.')


In [ ]:
if (adapter/'TRAINING_COMPLETE').is_file():
    output = RUN_ROOT / f'{adapter.name}_evalcrop{CROPS}.jsonl'
    run_module('baseline.infer', '--model', TRAIN_MODEL, '--setting', 'downstream', '--adapter', adapter,
               '--data', DATA[TRAIN_DATASET]['test'], '--image-root', DATA[TRAIN_DATASET]['images'],
               '--crops', CROPS, '--max-new-tokens', MAX_NEW_TOKENS, '--output', output,
               '--label-samples', LABEL_SAMPLES)
    run_module('baseline.evaluate', '--data', DATA[TRAIN_DATASET]['test'], '--predictions', output,
               '--output', RUN_ROOT/f'{output.stem}_report', '--plots')
else:
    print('학습 완료 checkpoint가 없습니다. downstream 평가 미실행.')


## 출력 확인 / calibration 확장

`comparison.md`: 논문값과 실행값/차이. `report.json`: coverage·모든 응답·ROC/sensitivity/specificity·answer별 entropy·low-entropy 오류·ECE 계열. `*.meta.json`: 모델 SHA/버전/seed/양자화/crop/data hash.

Temperature scaling은 test에 fit하지 않습니다. 별도 validation JSONL로 `baseline.infer --split validation ...` 실행 후 `baseline.evaluate --validation-predictions ...`를 추가하세요. train에서 validation을 떼었다면 학습에서도 그 샘플을 반드시 제외하세요. validation이 없으면 ECE-t/BS-t는 N/A입니다.

RAG·Expert-CFG sweep, 외부 7B/13B/32B 모델은 선택한 범위 밖입니다. 이들도 포함한 논문 전체 실험 방법과 미공개 사항은 docs/EXPERIMENT_AUDIT.md에 모두 기록되어 있습니다. GPU 결과를 얻기 전에는 재현 성공으로 보고하지 마세요.


In [ ]:
from IPython.display import display, Markdown
for report in sorted(RUN_ROOT.glob('*_report/comparison.md')):
    print(report)
    display(Markdown(report.read_text()))
